In [1]:
!pip install gradio -q

In [2]:
!pip install torch torchvision huggingface_hub -q

In [3]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import json
import gradio as gr
#from huggingface_hub import Repository, login
#from huggingface_hub import hf_hub_download

In [4]:
from torchvision.models import resnet50
from models import resnet50

# define checkpoint path

In [5]:
#model = resnet50()
model_path = "/content/checkpoint-epcoh24_lr_point1.pth"

# input image preprocessing as per data_module during training

In [6]:
def preprocess_image(image):
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    image = image.convert("RGB")  # Ensure image is in RGB format
    image = preprocess(image)  # Apply transformations
    image = image.unsqueeze(0)  # Add batch dimension
    return image

# Output defining

In [7]:
# Prediction function
def predict(image):
    image_tensor = preprocess_image(image)
    with torch.no_grad():
        output = model(image_tensor)
        probabilities = torch.nn.functional.softmax(output, dim=1)
        top5_probabilities, top5_indices = probabilities.topk(5)

    results = {}
    for i in range(5):
        class_index = top5_indices[0][i].item()
        class_label = class_labels.get(str(class_index), "Unknown class")
        results[class_label] = top5_probabilities[0][i].item()  # Store label and probability in a dictionary
    print("See the prediction result : ", results)
    return results  # Return the results as a dictionary

# load the lables from json

In [8]:
# Load class index mapping
def load_class_labels(label_path):
    with open(label_path, 'r') as f:
        class_labels = json.load(f)
    return class_labels

# load model

In [9]:
def load_model(model_path):
    model = resnet50()  # Create an instance of the ResNet-50 model
    checkpoint = torch.load(model_path, map_location='cpu')  # Load the checkpoint

    # Access the 'model_state_dict' key within the checkpoint
    state_dict = checkpoint['model_state_dict']

    # Remove "module." prefix if present (common when using DataParallel)
    if 'module.' in next(iter(state_dict.keys())):
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

    model.load_state_dict(state_dict)  # Load the state dictionary
    model.eval()  # Set the model to evaluation mode
    return model

In [10]:
# Load model and class labels
# model_path = 'model.pt'  # Path to the trained model
model_path = "/content/checkpoint-epcoh24_lr_point1.pth"
label_path = '/content/imagenet_class_index.json'  # Path to the class index mapping
model = load_model(model_path)
class_labels = load_class_labels(label_path)

<ipython-input-9-9fb24dfa1c7e>:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location='cpu')  # Load the checkpoint


# create gradio interface

In [11]:
# Create Gradio interface
iface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=5),
    title="Image Classification using ResNet 50 Model trained on Imagenet 1000 dataset",
    description="Upload an image to get the top-5 predictions."
)



# Launch Gradio on web

In [12]:
# Launch the app
iface.launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1a7decd249108c2d2e.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
